[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap04/cap04.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

Esta lista transforma os conceitos do Capítulo 4 em uma trilha prática de segmentação e morfologia matemática. Os EPs começam com limiarização e avançam até rotulação e descritores de componentes, sempre com matrizes pequenas para que cada pixel possa ser conferido à mão.

::: {.callout-important}
### Regra comum dos EPs morfológicos {.unnumbered}

Nas operações com vizinhança, **não faça padding**. Para cada pixel, avalie apenas as posições do elemento estruturante que caem dentro do domínio da imagem. Essa é a mesma ideia das implementações didáticas em `morph.py`, como `mm.dil0`, `mm.ero0`, `mm.dil1` e `mm.label0`: a vizinhança é recortada pelo domínio válido da imagem.
:::


---

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de  **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [1]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.1 | TestSuite: 1.1.0


---

#### Executando os Testes {.unnumbered}

Para rodar os testes, execute `TestSuite("EP04_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

---

### EP04_01 🎚️ Limiarização Global para Máscara Binária

Uma câmera de inspeção industrial precisa transformar uma imagem em tons de cinza em uma máscara simples: peça ou fundo. Este EP fixa a ideia central da segmentação por limiar, que será usada como porta de entrada para os operadores morfológicos.

#### 📋 Diretrizes de Implementação

1. Ler $L$, $C$ e o limiar inteiro $T$.
2. Ler a matriz $f$ com intensidades em $[0,255]$.
3. Para cada pixel, gerar uma máscara binária:

$$
g(y,x)=
\begin{cases}
255, & f(y,x) \ge T \\
0,   & f(y,x) < T
\end{cases}
$$

4. Imprimir a matriz binária resultante.

#### 📌 Restrições Computacionais

* Não use bibliotecas de processamento de imagem.
* A comparação é **maior ou igual** ao limiar.
* A saída usa apenas `0` e `255`.

#### 🧠 Fundamentação Teórica

A limiarização converte uma imagem de intensidade em uma partição binária. Ela é simples, rápida e serve como primeiro passo para limpeza morfológica, rotulação e contagem de objetos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: inteiro $L$.
* Linha 2: inteiro $C$.
* Linha 3: inteiro $T$.
* Próximas $L$ linhas: matriz da imagem.

**Saída:**

* Matriz binária $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 255 255<br>255 0 255 0 | Pixels iguais a $T$ entram no objeto |
| 1<br>5<br>50<br>10 50 49 200 0 | 0 255 0 255 0 | Máscara binária direta |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [5]:
%%writefile EP04_01.py
L = int(input())
C = int(input())
T = int(input())

img = [list(map(int, input().split())) for _ in range(L)]

for y in range(L):
    row = [255 if img[y][x] >= T else 0 for x in range(C)]
    print(*row)


Writing EP04_01.py


In [6]:
TestSuite("EP04_01.py").run()


---

### EP04_02 📊 Limiar de Otsu Manual

Agora o sistema deve escolher o limiar sozinho. Você vai implementar o critério de Otsu sem chamar funções prontas, observando como a melhor separação entre fundo e objeto aparece no histograma.

#### 📋 Diretrizes de Implementação

1. Ler $L$, $C$ e a imagem.
2. Construir o histograma com 256 posições.
3. Para cada $T$ em $[0,254]$, separar as classes $C_0=\{p \le T\}$ e $C_1=\{p>T\}$.
4. Calcular $\sigma_B^2(T)=w_0w_1(\mu_0-\mu_1)^2$.
5. Escolher o menor $T$ que maximiza $\sigma_B^2$.
6. Imprimir $T$ e a máscara binária com `255` para pixels maiores que $T$.

#### 📌 Restrições Computacionais

* Ignore limiares que deixam uma das classes vazia.
* Em empate, escolha o menor limiar.
* Não use `cv2.threshold`, `skimage` ou funções prontas de Otsu.

#### 🧠 Fundamentação Teórica

O método de Otsu procura o limiar que maximiza a separação estatística entre duas classes. Quando o histograma é bimodal, essa escolha costuma produzir uma segmentação inicial muito boa.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: inteiro $L$.
* Linha 2: inteiro $C$.
* Próximas $L$ linhas: matriz da imagem.

**Saída:**

* Linha 1: limiar ótimo $T$.
* Próximas $L$ linhas: máscara com `0` e `255`, usando a regra $p>T$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>4<br>0 0 10 10<br>200 200 210 210 | 10<br>0 0 0 0<br>255 255 255 255 | O melhor corte separa os dois grupos |
| 1<br>6<br>5 5 5 100 100 100 | 5<br>0 0 0 255 255 255 | Dois picos bem separados |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [7]:
%%writefile EP04_02.py
L = int(input())
C = int(input())
img = [list(map(int, input().split())) for _ in range(L)]
vals = [p for row in img for p in row]
N = len(vals)

hist = [0] * 256
for p in vals:
    hist[p] += 1

total_sum = sum(i * hist[i] for i in range(256))
w0 = 0
sum0 = 0
best_T = 0
best_var = -1.0

for T in range(255):
    w0 += hist[T]
    sum0 += T * hist[T]
    w1 = N - w0
    if w0 == 0 or w1 == 0:
        continue
    mu0 = sum0 / w0
    mu1 = (total_sum - sum0) / w1
    var_between = w0 * w1 * (mu0 - mu1) ** 2
    if var_between > best_var:
        best_var = var_between
        best_T = T

print(best_T)
for y in range(L):
    print(*[255 if img[y][x] > best_T else 0 for x in range(C)])


Writing EP04_02.py


In [8]:
TestSuite("EP04_02.py").run()


---

### EP04_03 ➕ Dilatação Plana Sem Padding

A dilatação expande regiões claras e propaga máximos locais. Neste EP, você implementa a versão didática equivalente à ideia de `mm.dil0`, respeitando a vizinhança válida de cada pixel.

#### 📋 Diretrizes de Implementação

1. Ler a imagem $f$ e o elemento estruturante binário $B$.
2. Refletir $B$ antes de aplicar a dilatação, isto é, usar $\hat{B}$.
3. Para cada pixel, considerar apenas vizinhos dentro da imagem e posições de $\hat{B}$ iguais a `1`.
4. Atribuir ao pixel de saída o maior valor encontrado.

#### 📌 Restrições Computacionais

* Não faça padding.
* O elemento estruturante terá dimensões ímpares e pelo menos um valor `1`.
* Trabalhe com inteiros e não altere as dimensões da imagem.

#### 🧠 Fundamentação Teórica

Na dilatação plana em tons de cinza, o novo valor é o máximo local definido pelo elemento estruturante. Em imagens binárias, isso corresponde a expandir o objeto.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante $B$.
* Próximas $L$ linhas: imagem $f$.

**Saída:**

* Matriz dilatada $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | O máximo se propaga em cruz |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [9]:
%%writefile EP04_03.py
def vizinhos(H, W, B, y, x, apenas_um=True):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W:
                if (not apenas_um) or B[by][bx] == 1:
                    yield vy, vx, B[by][bx]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

B_ref = [row[::-1] for row in B[::-1]]
g = []
for y in range(L):
    row = []
    for x in range(C):
        row.append(max(f[vy][vx] for vy, vx, _ in vizinhos(L, C, B_ref, y, x)))
    g.append(row)

for row in g:
    print(*row)


Writing EP04_03.py


In [10]:
TestSuite("EP04_03.py").run()


---

### EP04_04 ➖ Erosão Plana Sem Padding

A erosão faz o movimento complementar: ela contrai regiões claras e propaga mínimos locais. Este EP consolida o raciocínio pixel a pixel usado em `mm.ero0`.

#### 📋 Diretrizes de Implementação

1. Ler a imagem $f$ e o elemento estruturante binário $B$.
2. Para cada pixel, visitar apenas posições válidas da vizinhança.
3. Usar somente posições de $B$ iguais a `1`.
4. Atribuir ao pixel de saída o menor valor encontrado.

#### 📌 Restrições Computacionais

* Não faça padding.
* Não assuma pixels externos iguais a zero.
* O resultado preserva as dimensões da entrada.

#### 🧠 Fundamentação Teórica

Na erosão plana em tons de cinza, o novo valor é o mínimo local definido por $B$. Em imagens binárias, um pixel permanece no objeto apenas quando a vizinhança exigida cabe dentro do objeto observado.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante $B$.
* Próximas $L$ linhas: imagem $f$.

**Saída:**

* Matriz erodida $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | O mínimo se propaga em cruz |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [11]:
%%writefile EP04_04.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

g = []
for y in range(L):
    row = []
    for x in range(C):
        row.append(min(f[vy][vx] for vy, vx in vizinhos(L, C, B, y, x)))
    g.append(row)

for row in g:
    print(*row)


Writing EP04_04.py


In [12]:
TestSuite("EP04_04.py").run()


---

### EP04_05 ⛰️ Dilatação em Tons de Cinza com Pesos

Nem todo elemento estruturante é plano. Em relevo topográfico, pesos positivos podem favorecer certas direções ou alturas. Aqui você implementa a ideia de `mm.dil1`: máximo de vizinhos somado ao peso correspondente.

#### 📋 Diretrizes de Implementação

1. Ler a imagem $f$ e uma máscara ponderada $b$.
2. Para cada pixel, percorrer todas as posições válidas de $b$.
3. Calcular $f(v_y,v_x)+b(b_y,b_x)$.
4. A saída recebe o maior desses valores.

#### 📌 Restrições Computacionais

* Não faça padding.
* Use todas as posições válidas de $b$, inclusive pesos zero ou negativos.
* Não aplique saturação, a menos que o enunciado de teste indique valores já limitados.

#### 🧠 Fundamentação Teórica

A dilatação não plana adiciona uma função estruturante ao sinal antes da busca pelo máximo. Ela modela uma sonda com relevo, não apenas uma janela plana.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_b$.
* Linha 4: $W_b$.
* Próximas $H_b$ linhas: máscara ponderada $b$.
* Próximas $L$ linhas: imagem $f$.

**Saída:**

* Matriz resultante $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92 | O peso central e os vizinhos competem pelo máximo |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [13]:
%%writefile EP04_05.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W:
                yield vy, vx, B[by][bx]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
b = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

for y in range(L):
    row = []
    for x in range(C):
        row.append(max(f[vy][vx] + peso for vy, vx, peso in vizinhos(L, C, b, y, x)))
    print(*row)


Writing EP04_05.py


In [14]:
TestSuite("EP04_05.py").run()


---

### EP04_06 🧹 Abertura Binária: Removendo Ruídos Claros

Um sensor detectou pequenos pontos falsos além do objeto principal. A abertura, erosão seguida de dilatação, é uma ferramenta clássica para remover ruídos brilhantes menores que o elemento estruturante.

#### 📋 Diretrizes de Implementação

1. Ler uma imagem binária com valores `0` e `1`.
2. Implementar erosão plana com $B$.
3. Implementar dilatação plana com $B$ refletido.
4. Calcular $A \circ B = (A \ominus B) \oplus B$.

#### 📌 Restrições Computacionais

* Não faça padding em nenhuma etapa.
* A saída deve usar `0` e `1`.
* Não use bibliotecas de morfologia prontas.

#### 🧠 Fundamentação Teórica

A abertura é anti-extensiva: ela não cria novas partes do objeto além daquelas sustentadas pela erosão. Por isso é ótima para limpar pontos pequenos de fundo.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante binário.
* Próximas $L$ linhas: imagem binária.

**Saída:**

* Imagem aberta $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>5<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>1 0 0 0 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 0 0 0 1 | 0 0 0 0 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 0 0 0 0 | Os pontos isolados desaparecem |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [15]:
%%writefile EP04_06.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erosao(f, B):
    H, W = len(f), len(f[0])
    return [[min(f[vy][vx] for vy, vx in vizinhos(H, W, B, y, x)) for x in range(W)] for y in range(H)]

def dilatacao(f, B):
    H, W = len(f), len(f[0])
    Bref = [row[::-1] for row in B[::-1]]
    return [[max(f[vy][vx] for vy, vx in vizinhos(H, W, Bref, y, x)) for x in range(W)] for y in range(H)]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

g = dilatacao(erosao(f, B), B)
for row in g:
    print(*row)


Writing EP04_06.py


In [16]:
TestSuite("EP04_06.py").run()


---

### EP04_07 🧩 Fechamento Binário: Preenchendo Falhas

Depois de segmentar uma peça, pequenas falhas aparecem dentro do objeto. O fechamento, dilatação seguida de erosão, ajuda a costurar lacunas e preencher buracos compatíveis com o elemento estruturante.

#### 📋 Diretrizes de Implementação

1. Ler uma imagem binária e $B$.
2. Aplicar dilatação plana.
3. Aplicar erosão plana no resultado.
4. Imprimir $A ullet B = (A \oplus B) \ominus B$.

#### 📌 Restrições Computacionais

* Não faça padding.
* Use a mesma regra de vizinhança válida nas duas etapas.
* A saída deve usar `0` e `1`.

#### 🧠 Fundamentação Teórica

O fechamento é extensivo em muitas situações práticas: ele preserva regiões maiores enquanto fecha pequenas interrupções internas ou estreitos vãos escuros.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante binário.
* Próximas $L$ linhas: imagem binária.

**Saída:**

* Imagem fechada $L 	imes C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 0 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | O buraco central é fechado |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [17]:
%%writefile EP04_07.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erosao(f, B):
    H, W = len(f), len(f[0])
    return [[min(f[vy][vx] for vy, vx in vizinhos(H, W, B, y, x)) for x in range(W)] for y in range(H)]

def dilatacao(f, B):
    H, W = len(f), len(f[0])
    Bref = [row[::-1] for row in B[::-1]]
    return [[max(f[vy][vx] for vy, vx in vizinhos(H, W, Bref, y, x)) for x in range(W)] for y in range(H)]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

g = erosao(dilatacao(f, B), B)
for row in g:
    print(*row)


Writing EP04_07.py


In [18]:
TestSuite("EP04_07.py").run()


---

### EP04_08 🌋 Gradiente Morfológico em Tons de Cinza

Bordas aparecem onde há diferença forte entre máximo e mínimo local. O gradiente morfológico transforma essa ideia em um detector simples e robusto de contornos.

#### 📋 Diretrizes de Implementação

1. Ler imagem em tons de cinza e elemento estruturante binário.
2. Calcular a dilatação plana.
3. Calcular a erosão plana.
4. Imprimir `dilatação - erosão` para cada pixel.

#### 📌 Restrições Computacionais

* Não faça padding.
* Use inteiros.
* Como a dilatação é maior ou igual à erosão, a diferença não precisa de saturação inferior.

#### 🧠 Fundamentação Teórica

O gradiente morfológico é definido por $
abla_B(f)=(f\oplus B)-(f\ominus B)$. Ele realça transições locais e é muito usado antes de segmentação por regiões.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante binário.
* Próximas $L$ linhas: imagem.

**Saída:**

* Matriz do gradiente morfológico.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10<br>10 80 10<br>10 10 10 | 70 70 70<br>70 70 70<br>70 70 70 | A diferença máximo-mínimo destaca o pico |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [19]:
%%writefile EP04_08.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erosao(f, B):
    H, W = len(f), len(f[0])
    return [[min(f[vy][vx] for vy, vx in vizinhos(H, W, B, y, x)) for x in range(W)] for y in range(H)]

def dilatacao(f, B):
    H, W = len(f), len(f[0])
    Bref = [row[::-1] for row in B[::-1]]
    return [[max(f[vy][vx] for vy, vx in vizinhos(H, W, Bref, y, x)) for x in range(W)] for y in range(H)]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

d = dilatacao(f, B)
e = erosao(f, B)
for y in range(L):
    print(*[d[y][x] - e[y][x] for x in range(C)])


Writing EP04_08.py


In [20]:
TestSuite("EP04_08.py").run()


---

### EP04_09 🗺️ Transformada de Distância por Erosões

A transformada de distância revela o “miolo” dos objetos: pixels mais internos recebem valores maiores. Ela aparece no capítulo como etapa importante para separar objetos grudados e preparar marcadores para segmentação por regiões.

#### 📋 Diretrizes de Implementação

1. Ler uma imagem binária e um elemento estruturante $B$.
2. Começar com uma matriz de distâncias zerada.
3. Enquanto ainda houver pixels `1`, registrar o nível atual nos pixels sobreviventes.
4. Erodir a imagem atual usando $B$, sem padding.
5. Imprimir o mapa de distâncias.

#### 📌 Restrições Computacionais

* Não faça padding.
* A distância do fundo é `0`.
* Pixels do objeto na primeira camada recebem distância `1`.
* Os casos de teste terão objetos com fundo alcançável dentro do domínio da imagem.

#### 🧠 Fundamentação Teórica

Erosões sucessivas descascam o objeto camada por camada. O número da última camada em que um pixel permanece ativo funciona como uma distância morfológica induzida pelo elemento estruturante. Com uma cruz 3×3, a métrica se aproxima da distância city-block.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: $H_B$.
* Linha 4: $W_B$.
* Próximas $H_B$ linhas: elemento estruturante binário.
* Próximas $L$ linhas: imagem binária.

**Saída:**

* Mapa de distâncias $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>5<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0 0 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 1 1 1 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 1 1 1 0<br>0 1 2 1 0<br>0 1 1 1 0<br>0 0 0 0 0 | O centro sobrevive a duas erosões |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [21]:
%%writefile EP04_09.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erosao(f, B):
    H, W = len(f), len(f[0])
    return [[min(f[vy][vx] for vy, vx in vizinhos(H, W, B, y, x)) for x in range(W)] for y in range(H)]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

dist = [[0] * C for _ in range(L)]
atual = [row[:] for row in f]
nivel = 0

while any(any(row) for row in atual):
    nivel += 1
    for y in range(L):
        for x in range(C):
            if atual[y][x] == 1:
                dist[y][x] = nivel
    proxima = erosao(atual, B)
    if proxima == atual:
        break
    atual = proxima

for row in dist:
    print(*row)


Writing EP04_09.py


In [22]:
TestSuite("EP04_09.py").run()


---

### EP04_10 🪙 Contagem e Descritores de Componentes

Para fechar a lista, você vai transformar uma máscara em uma pequena tabela de objetos. É a base de pipelines que contam moedas, partículas, células ou defeitos industriais.

#### 📋 Diretrizes de Implementação

1. Ler uma imagem binária e conectividade $K$.
2. Rotular os componentes como no EP anterior.
3. Para cada componente, calcular área e caixa delimitadora.
4. Imprimir a quantidade de componentes e uma linha por componente.

#### 📌 Restrições Computacionais

* A área é o número de pixels do componente.
* A caixa deve usar índices baseados em zero: `min_linha min_coluna max_linha max_coluna`.
* Liste os componentes na ordem crescente dos rótulos.

#### 🧠 Fundamentação Teórica

Descritores de forma são medidas compactas sobre regiões segmentadas. Área e caixa delimitadora são simples, mas já permitem contagem, filtragem por tamanho e inspeção de localização.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: $L$.
* Linha 2: $C$.
* Linha 3: conectividade $K$ (`4` ou `8`).
* Próximas $L$ linhas: imagem binária com `0` e `1`.

**Saída:**

* Linha 1: número de componentes $N$.
* Próximas $N$ linhas: `rotulo area min_linha min_coluna max_linha max_coluna`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 4<br>6<br>4<br>1 1 0 0 1 0<br>1 0 0 0 1 1<br>0 0 1 0 0 0<br>0 0 1 1 0 1 | 4<br>1 3 0 0 1 1<br>2 3 0 4 1 5<br>3 3 2 2 3 3<br>4 1 3 5 3 5 | Cada objeto vira uma linha de tabela |
---

#### 📌 Observação de Formatação

* A saída deve conter apenas os valores pedidos, sem textos extras.
* Em matrizes, imprima uma linha por linha da imagem, com valores separados por um espaço.


In [23]:
%%writefile EP04_10.py
from collections import deque

L = int(input())
C = int(input())
K = int(input())
f = [list(map(int, input().split())) for _ in range(L)]

if K == 4:
    dirs = [(-1, 0), (0, -1), (0, 1), (1, 0)]
else:
    dirs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

labels = [[0] * C for _ in range(L)]
desc = []
current = 0

for y in range(L):
    for x in range(C):
        if f[y][x] == 1 and labels[y][x] == 0:
            current += 1
            q = deque([(y, x)])
            labels[y][x] = current
            area = 0
            min_y = max_y = y
            min_x = max_x = x
            while q:
                cy, cx = q.popleft()
                area += 1
                min_y = min(min_y, cy)
                max_y = max(max_y, cy)
                min_x = min(min_x, cx)
                max_x = max(max_x, cx)
                for dy, dx in dirs:
                    ny, nx = cy + dy, cx + dx
                    if 0 <= ny < L and 0 <= nx < C:
                        if f[ny][nx] == 1 and labels[ny][nx] == 0:
                            labels[ny][nx] = current
                            q.append((ny, nx))
            desc.append((current, area, min_y, min_x, max_y, max_x))

print(len(desc))
for item in desc:
    print(*item)


Writing EP04_10.py


In [24]:
TestSuite("EP04_10.py").run()
